# Notebook de Forecasting - Inferencia

In [3]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays

In [4]:
# Cargar datos de inferencia
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')

# Mostrar información básica
print(f'Shape del dataset: {inferencia_df.shape}')
print(f'\nPrimeras filas:')
inferencia_df.head()

Shape del dataset: (888, 13)

Primeras filas:


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [19]:
# Cargar datos de inferencia
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')

print(f'Shape inicial: {inferencia_df.shape}')
print(f'\nPrimeras filas:')
display(inferencia_df.head())

# Verificar estructura
print(f'\nVerificación:')
print(f'  - Registros totales: {len(inferencia_df)}')
print(f'  - Productos únicos: {inferencia_df["producto_id"].nunique()}')
print(f'  - Registros CON unidades_vendidas: {inferencia_df["unidades_vendidas"].notna().sum()}')
print(f'  - Registros SIN unidades_vendidas (NaN): {inferencia_df["unidades_vendidas"].isna().sum()}')

Shape inicial: (888, 13)

Primeras filas:


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41



Verificación:
  - Registros totales: 888
  - Productos únicos: 24
  - Registros CON unidades_vendidas: 168
  - Registros SIN unidades_vendidas (NaN): 720


In [20]:
# Variables temporales y de calendario
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek
inferencia_df['nombre_dia_semana'] = inferencia_df['fecha'].dt.day_name()
inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['es_fin_semana'] = inferencia_df['dia_semana'].isin([5,6])

# Festivos
ar_holidays = holidays.country_holidays('AR', years=inferencia_df['año'].unique())
inferencia_df['ar_festivo'] = inferencia_df['fecha'].isin(ar_holidays)

# Black Friday y Cyber Monday
def es_black_friday(fecha):
    if fecha.month == 11 and fecha.weekday() == 4:
        last_friday = max([d for d in pd.date_range(fecha.replace(day=1), fecha.replace(day=30)) if d.weekday() == 4])
        return fecha == last_friday
    return False

def es_cyber_monday(fecha):
    if fecha.month == 11 and fecha.weekday() == 0:
        first_monday = min([d for d in pd.date_range(fecha.replace(day=1), fecha.replace(day=7)) if d.weekday() == 0])
        return fecha == first_monday
    return False

inferencia_df['ar_Black_Friday'] = inferencia_df['fecha'].apply(es_black_friday)
inferencia_df['ar_Cyber_Monday'] = inferencia_df['fecha'].apply(es_cyber_monday)
inferencia_df['es_laborable'] = (~inferencia_df['es_fin_semana']) & (~inferencia_df['ar_festivo'])
inferencia_df['mitad_mes'] = inferencia_df['dia_mes'].between(10,20)
inferencia_df['inicio_mes'] = inferencia_df['dia_mes'] <= 7
inferencia_df['fin_mes'] = inferencia_df['dia_mes'] >= 25

print('✓ Variables temporales creadas')
print(f'  Total registros: {len(inferencia_df)}')

✓ Variables temporales creadas
  Total registros: 888


C:\Users\x\AppData\Local\Temp\ipykernel_17092\2748513730.py:14: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['ar_festivo'] = inferencia_df['fecha'].isin(ar_holidays)


In [21]:
# Crear LAGs usando TODOS los datos (octubre + noviembre)
# Los lags de noviembre se calcularán usando octubre

# Ordenar
inferencia_df = inferencia_df.sort_values(['producto_id', 'año', 'fecha']).reset_index(drop=True)

print(f'Antes de crear lags: {len(inferencia_df)} registros')

# Crear LAGs del 1 al 7
for lag in range(1, 8):
    inferencia_df[f'unidades_vendidas_lag{lag}'] = inferencia_df.groupby(['producto_id', 'año'])['unidades_vendidas'].shift(lag)

# Media móvil
inferencia_df['unidades_vendidas_mm7'] = inferencia_df.groupby(['producto_id', 'año'])['unidades_vendidas'].transform(lambda x: x.rolling(7).mean())

print(f'Después de crear lags: {len(inferencia_df)} registros')
print('✓ LAGs creados - NO SE ELIMINÓ NINGUNA FILA')

Antes de crear lags: 888 registros
Después de crear lags: 888 registros
✓ LAGs creados - NO SE ELIMINÓ NINGUNA FILA


In [22]:
# Filtrar SOLO noviembre
print(f'\nAntes del filtro:')
print(f'  - Total: {len(inferencia_df)}')
print(f'  - Meses: {sorted(inferencia_df["mes"].unique())}')

# FILTRAR
inferencia_df = inferencia_df[inferencia_df['mes'] == 11].copy()

print(f'\nDespués del filtro:')
print(f'  - Total: {len(inferencia_df)}')
print(f'  - Meses: {sorted(inferencia_df["mes"].unique())}')
print(f'  - Fechas: {inferencia_df["fecha"].min()} a {inferencia_df["fecha"].max()}')

# Verificar LAGs (deben tener valores de octubre)
print(f'\nVerificación de LAGs:')
print(f'  - lag1 con valores: {inferencia_df["unidades_vendidas_lag1"].notna().sum()}/{len(inferencia_df)}')
print(f'  - lag7 con valores: {inferencia_df["unidades_vendidas_lag7"].notna().sum()}/{len(inferencia_df)}')
print(f'  - mm7 con valores: {inferencia_df["unidades_vendidas_mm7"].notna().sum()}/{len(inferencia_df)}')

# Mostrar muestra
print(f'\nMuestra de datos:')
display(inferencia_df[['fecha', 'producto_id', 'unidades_vendidas', 'unidades_vendidas_lag1', 'unidades_vendidas_lag7']].head(10))


Antes del filtro:
  - Total: 888
  - Meses: [np.int32(10), np.int32(11)]

Después del filtro:
  - Total: 720
  - Meses: [np.int32(11)]
  - Fechas: 2025-11-01 00:00:00 a 2025-11-30 00:00:00

Verificación de LAGs:
  - lag1 con valores: 24/720
  - lag7 con valores: 168/720
  - mm7 con valores: 0/720

Muestra de datos:


,fecha,producto_id,unidades_vendidas,unidades_vendidas_lag1,unidades_vendidas_lag7
7,2025-11-01,PROD_001,NaN,14.0,26.0
8,2025-11-02,PROD_001,NaN,NaN,20.0
9,2025-11-03,PROD_001,NaN,NaN,16.0
10,2025-11-04,PROD_001,NaN,NaN,15.0
11,2025-11-05,PROD_001,NaN,NaN,14.0
12,2025-11-06,PROD_001,NaN,NaN,10.0
13,2025-11-07,PROD_001,NaN,NaN,14.0
14,2025-11-08,PROD_001,NaN,NaN,NaN
15,2025-11-09,PROD_001,NaN,NaN,NaN
16,2025-11-10,PROD_001,NaN,NaN,NaN


In [23]:
# Descuento porcentaje
inferencia_df['descuento_porcentaje'] = ((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base']) * 100
inferencia_df['descuento_porcentaje'] = inferencia_df['descuento_porcentaje'].round(2)

# Precio competencia
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']

print('✓ Descuento y precios calculados')

✓ Descuento y precios calculados


In [24]:
# One-Hot Encoding
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

one_hot = pd.get_dummies(
    inferencia_df[['nombre_h', 'categoria_h', 'subcategoria_h']], 
    prefix=['nombre_h', 'categoria_h', 'subcategoria_h']
)
inferencia_df = pd.concat([inferencia_df, one_hot], axis=1)

print(f'✓ One-hot encoding aplicado')
print(f'  Shape final: {inferencia_df.shape}')

✓ One-hot encoding aplicado
  Shape final: (720, 86)


In [25]:
# Guardar
output_path = '../data/processed/inferencia_df_transformado.csv'
inferencia_df.to_csv(output_path, index=False)

print(f'\n{"="*60}')
print(f'✓ TRANSFORMACIÓN COMPLETADA')
print(f'{"="*60}')
print(f'Archivo: {output_path}')
print(f'Registros: {len(inferencia_df)}')
print(f'Columnas: {len(inferencia_df.columns)}')
print(f'Productos: {inferencia_df["producto_id"].nunique()}')
print(f'Periodo: {inferencia_df["fecha"].min()} a {inferencia_df["fecha"].max()}')
print(f'{"="*60}')


✓ TRANSFORMACIÓN COMPLETADA
Archivo: ../data/processed/inferencia_df_transformado.csv
Registros: 720
Columnas: 86
Productos: 24
Periodo: 2025-11-01 00:00:00 a 2025-11-30 00:00:00
